# CommGuard adversarial red-team v1

Bounded defensive robustness research. Execution is disabled by default and requires a hash-pinned benign acceptance archive plus explicit approval. The final family/session/config holdout remains sealed unless separately released.

> **Prototype scope:** CommGuard’s Kaggle workflow is a single-node, dual-NVIDIA-T4 research prototype. It validates experimental methodology and software behavior on two local GPU ranks. It does not establish generalization to two physical 8-GPU nodes, NVLink/NVSwitch fabrics, RoCE or InfiniBand networks, large frontier-model workloads, or production treaty-verification deployments.


In [ ]:
import hashlib
import importlib
import os
from pathlib import Path
import re
import subprocess
import sys

NOTEBOOK_VERSION = "commguard_adversarial_redteam_v1"
REPOSITORY_URL = "https://github.com/waqasm86/CommGuard.git"
INSTALL_SOURCE = "auto"  # auto: wheel, source archive, pinned Git commit, then dev source.
PINNED_PUBLIC_COMMIT = ""  # Required for public Git installation.
EXPECTED_PACKAGE_SHA256 = ""  # Required for a supplied wheel or source archive.
EXPECTED_NOTEBOOK_SHA256 = ""  # SHA-256 of this canonical source notebook.
DEVELOPMENT_SMOKE_TEST = False
DEVELOPMENT_SOURCE = Path("/kaggle/working/commguard-development-source")
REPOSITORY = Path("/kaggle/working/commguard-source")

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

wheel_candidates = sorted(Path("/kaggle/input").rglob("commguard*.whl"))
archive_candidates = sorted(
    path for path in Path("/kaggle/input").rglob("commguard*")
    if path.is_file() and path.name.endswith((".tar.gz", ".zip"))
    and not any(token in path.name for token in (
        "-prototype-", "review-bundle", "calibration", "benign-corpus",
        "detector-evaluation", "adversarial-redteam",
    ))
)
selected = None
method = INSTALL_SOURCE
if method == "auto":
    method = "wheel" if wheel_candidates else "archive" if archive_candidates else "git"
if method == "wheel":
    if len(wheel_candidates) != 1:
        raise RuntimeError(f"Expected exactly one CommGuard wheel, observed {wheel_candidates}")
    selected = wheel_candidates[0]
elif method == "archive":
    if len(archive_candidates) != 1:
        raise RuntimeError(
            f"Expected exactly one CommGuard source archive, observed {archive_candidates}"
        )
    selected = archive_candidates[0]

if selected is not None:
    actual_package_sha256 = file_sha256(selected)
    if not re.fullmatch(r"[0-9a-f]{64}", EXPECTED_PACKAGE_SHA256):
        raise RuntimeError("Set EXPECTED_PACKAGE_SHA256 for the supplied package.")
    if actual_package_sha256 != EXPECTED_PACKAGE_SHA256:
        raise RuntimeError("Supplied package SHA-256 does not match.")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--no-deps", str(selected)], check=True
    )
    SOURCE_IDENTITY = f"sha256:{actual_package_sha256}"
    SOURCE_DIRTY = False
    INSTALL_PROVENANCE = {
        "install_source": method,
        "install_path": str(selected),
        "package_sha256": actual_package_sha256,
        "source_identity": SOURCE_IDENTITY,
    }
elif method == "git":
    if not re.fullmatch(r"[0-9a-f]{40}", PINNED_PUBLIC_COMMIT):
        raise RuntimeError("Set PINNED_PUBLIC_COMMIT to a pushed 40-character commit.")
    if not REPOSITORY.exists():
        subprocess.run(
            [
                "git", "clone", "--filter=blob:none", "--no-checkout",
                REPOSITORY_URL, str(REPOSITORY),
            ],
            check=True,
        )
    if not (REPOSITORY / ".git").is_dir():
        raise RuntimeError(f"Refusing non-Git source directory: {REPOSITORY}")
    subprocess.run(
        ["git", "-C", str(REPOSITORY), "fetch", "origin", PINNED_PUBLIC_COMMIT],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPOSITORY), "checkout", "--detach", PINNED_PUBLIC_COMMIT], check=True
    )
    head = subprocess.run(
        ["git", "-C", str(REPOSITORY), "rev-parse", "HEAD"], check=True,
        capture_output=True, text=True,
    ).stdout.strip()
    dirty = subprocess.run(
        ["git", "-C", str(REPOSITORY), "status", "--porcelain"], check=True,
        capture_output=True, text=True,
    ).stdout.strip()
    pushed_refs = subprocess.run(
        ["git", "-C", str(REPOSITORY), "branch", "-r", "--contains", head], check=True,
        capture_output=True, text=True,
    ).stdout.strip()
    if head != PINNED_PUBLIC_COMMIT or dirty or not pushed_refs:
        raise RuntimeError("Pinned Git source is dirty, mismatched, or not remote-visible.")
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "--no-build-isolation",
            "--no-deps", str(REPOSITORY),
        ],
        check=True,
    )
    SOURCE_IDENTITY = head
    SOURCE_DIRTY = False
    INSTALL_PROVENANCE = {
        "install_source": "pinned_public_git_commit",
        "repository_url": REPOSITORY_URL,
        "source_identity": head,
        "remote_refs": pushed_refs.splitlines(),
    }
elif method == "development":
    if not DEVELOPMENT_SMOKE_TEST or not (DEVELOPMENT_SOURCE / "pyproject.toml").is_file():
        raise RuntimeError(
            "Editable development source is allowed only for an explicit smoke test."
        )
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "--no-build-isolation",
            "--no-deps", "-e", str(DEVELOPMENT_SOURCE),
        ],
        check=True,
    )
    SOURCE_IDENTITY = "development-editable"
    SOURCE_DIRTY = True
    INSTALL_PROVENANCE = {
        "install_source": "editable_local_development",
        "source_identity": SOURCE_IDENTITY,
        "development_smoke_only": True,
    }
else:
    raise RuntimeError(f"Unsupported INSTALL_SOURCE={method!r}")

importlib.invalidate_caches()
for module_name in [
    name for name in sys.modules if name == "commguard" or name.startswith("commguard.")
]:
    del sys.modules[module_name]
import commguard
REVIEWED_COMMIT = SOURCE_IDENTITY
PIP_FREEZE = subprocess.run(
    [sys.executable, "-m", "pip", "freeze"], check=True, capture_output=True, text=True
).stdout.splitlines()
INSTALL_PROVENANCE.update({
    "commguard_version": commguard.__version__,
    "commguard_import": str(Path(commguard.__file__).resolve()),
    "python_version": sys.version,
    "pip_freeze": PIP_FREEZE,
})
print({
    "source_identity": SOURCE_IDENTITY,
    "source_dirty": SOURCE_DIRTY,
    "installation": INSTALL_PROVENANCE,
})


In [ ]:
from commguard.artifacts import restore_archive, sha256_file

INPUT_ARCHIVE = Path("/kaggle/input/commguard-detector-evaluation-prototype/commguard-detector-evaluation-prototype-REPLACE.tar.gz")
EXPECTED_INPUT_SHA256 = ""  # Required: SHA-256 printed by the preceding notebook.
ARTIFACTS = Path("/kaggle/working/commguard-artifacts")

if not re.fullmatch(r"[0-9a-f]{64}", EXPECTED_INPUT_SHA256):
    raise RuntimeError("Set EXPECTED_INPUT_SHA256 to the exact 64-character archive hash.")
actual_input_sha256 = sha256_file(INPUT_ARCHIVE)
if actual_input_sha256 != EXPECTED_INPUT_SHA256:
    raise RuntimeError(
        "Input archive hash mismatch: "
        f"expected={EXPECTED_INPUT_SHA256} actual={actual_input_sha256}"
    )
restore_archive(INPUT_ARCHIVE, ARTIFACTS, expected_sha256=EXPECTED_INPUT_SHA256)
print({"restored_archive": str(INPUT_ARCHIVE), "sha256": actual_input_sha256})


In [ ]:
from datetime import datetime, timezone

NOTEBOOK_RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")


import socket
from commguard.environment.preflight import check_environment, summarize_environment
from commguard.provenance import ProvenanceContext

NOTEBOOK_FILENAME = f"{NOTEBOOK_VERSION}.ipynb"
if not re.fullmatch(r"[0-9a-f]{64}", EXPECTED_NOTEBOOK_SHA256):
    raise RuntimeError("Set EXPECTED_NOTEBOOK_SHA256 to the canonical notebook source hash.")
if (REPOSITORY / "notebooks" / NOTEBOOK_FILENAME).is_file():
    actual_notebook_sha256 = file_sha256(REPOSITORY / "notebooks" / NOTEBOOK_FILENAME)
    if actual_notebook_sha256 != EXPECTED_NOTEBOOK_SHA256:
        raise RuntimeError("Canonical notebook SHA-256 does not match the pinned Git source.")
DIRTY_SOURCE_SMOKE_ONLY = bool(SOURCE_DIRTY and DEVELOPMENT_SMOKE_TEST)
if SOURCE_DIRTY and not DIRTY_SOURCE_SMOKE_ONLY:
    raise RuntimeError("Dirty source cannot create accepted research evidence.")
CONTEXT = ProvenanceContext(
    corpus_id=f"corpus-adversarial-v1-{NOTEBOOK_RUN_ID}",
    experiment_session_id=f"session-adversarial-v1-{NOTEBOOK_RUN_ID}",
    collection_id=f"collection-adversarial-v1-{NOTEBOOK_RUN_ID}",
    node_id=socket.gethostname(),
    source_commit=SOURCE_IDENTITY,
    source_dirty=SOURCE_DIRTY,
    notebook_version="commguard_adversarial_redteam_v1",
    input_archive_sha256=EXPECTED_INPUT_SHA256,
    random_seed=20260730,
)
ENVIRONMENT = check_environment(strict=True, output=ARTIFACTS, provenance=CONTEXT)
print(summarize_environment(ENVIRONMENT))
print({
    "notebook_run_id": NOTEBOOK_RUN_ID,
    "experiment_session_id": CONTEXT.experiment_session_id,
    "collection_id": CONTEXT.collection_id,
    "corpus_id": CONTEXT.corpus_id,
    "source_commit": CONTEXT.source_commit,
    "source_dirty": CONTEXT.source_dirty,
    "notebook_sha256": EXPECTED_NOTEBOOK_SHA256,
    "development_smoke_only": DIRTY_SOURCE_SMOKE_ONLY,
    "input_archive_sha256": CONTEXT.input_archive_sha256,
})


In [ ]:
import json

BENIGN_EVALUATION_ARTIFACT_PATH = Path("results/evaluation-REPLACE.json")
if BENIGN_EVALUATION_ARTIFACT_PATH.is_absolute() or ".." in BENIGN_EVALUATION_ARTIFACT_PATH.parts:
    raise RuntimeError("Benign evaluation path must be artifact-root-relative.")
benign_evaluation_path = ARTIFACTS / BENIGN_EVALUATION_ARTIFACT_PATH
if not benign_evaluation_path.is_file():
    raise RuntimeError("Adversarial work requires the exact saved benign evaluation.")
BENIGN_ACCEPTANCE = json.loads(benign_evaluation_path.read_text(encoding="utf-8"))
if not BENIGN_ACCEPTANCE.get("coverage_gate", {}).get("passed"):
    raise RuntimeError("Adversarial work blocked: benign primary coverage did not pass.")
if not BENIGN_ACCEPTANCE.get("primary_communication_only"):
    raise RuntimeError("Adversarial work blocked: benign primary baseline is missing.")
BENIGN_EXTRACTION_SUMMARY = Path(BENIGN_ACCEPTANCE["feature_source"])
if BENIGN_EXTRACTION_SUMMARY.is_absolute():
    matches = sorted((ARTIFACTS / "features").glob(BENIGN_EXTRACTION_SUMMARY.name))
    if len(matches) != 1:
        raise RuntimeError(
            "Cannot resolve the benign extraction summary inside the restored archive."
        )
    BENIGN_EXTRACTION_SUMMARY = matches[0].relative_to(ARTIFACTS)
print({"accepted_benign_evaluation": str(benign_evaluation_path), "coverage": "passed"})


In [ ]:
from commguard.adversarial import ADVERSARIAL_STRATEGIES, AdversarialHoldoutPlan
from commguard.orchestrator import estimate_matrix

HOLDOUT_PLAN = AdversarialHoldoutPlan(
    development_families=(
        "gradient_accumulation",
        "periodic_local_sgd",
        "segmented_runs",
        "idle_padding",
    ),
    hardening_families=(
        "randomized_synchronization",
        "mixed_training_inference",
        "synthetic_communication_decoy",
    ),
    final_family="diloco_inspired",
    final_session_ids=("session-reserved-final-v1",),
    final_config_ids=("diloco-inspired-inner10-v1",),
)
HOLDOUT_PLAN.validate()
for strategy_id, strategy in sorted(ADVERSARIAL_STRATEGIES.items()):
    print({
        "strategy": strategy_id,
        "purpose": strategy.research_purpose,
        "claim_boundary": strategy.claim_boundary,
        "enabled_by_default": strategy.enabled_by_default,
    })
print({"estimate": estimate_matrix("standard", 3), "comparison_only": "benign matrix"})


In [ ]:
from commguard.orchestrator import run_periodic_synchronization_study

RUN_MODE = "smoke"  # "smoke" or "full"
if RUN_MODE not in {"smoke", "full"}:
    raise RuntimeError("RUN_MODE must be smoke or full.")
if SOURCE_DIRTY and RUN_MODE != "smoke":
    raise RuntimeError("Dirty editable source is restricted to RUN_MODE='smoke'.")
DEVELOPMENT_SMOKE_ONLY = RUN_MODE == "smoke" or DIRTY_SOURCE_SMOKE_ONLY
RUN_FULL_PERIODIC_SYNCHRONIZATION_STUDY = RUN_MODE == "full"
ADVERSARIAL_HUMAN_APPROVAL = False
SYNCHRONIZATION_INTERVALS = (1, 2, 4, 8, 16)

ADVERSARIAL_MATRIX = None
if RUN_FULL_PERIODIC_SYNCHRONIZATION_STUDY:
    if not ADVERSARIAL_HUMAN_APPROVAL:
        raise RuntimeError("Set ADVERSARIAL_HUMAN_APPROVAL only after human review.")
    ADVERSARIAL_MATRIX = run_periodic_synchronization_study(
        output=ARTIFACTS,
        holdout_plan=HOLDOUT_PLAN,
        adversarial_approval=ADVERSARIAL_HUMAN_APPROVAL,
        synchronization_intervals=SYNCHRONIZATION_INTERVALS,
        repetitions=1,
        timeout_s=180.0,
        provenance=CONTEXT,
    )
    for family, row in sorted(ADVERSARIAL_MATRIX["family_counts"].items()):
        print({"family": family, **row})
else:
    from commguard.artifacts import ArtifactStore
    from commguard.scope import with_prototype_scope

    ArtifactStore(ARTIFACTS).write_json(
        "prototype/adversarial/run_status.json",
        with_prototype_scope({
            "execution_state": "development_smoke_plan_only",
            "development_smoke_only": True,
            "scientific_acceptance_eligible": False,
            "bounded_research_redteam": True,
            "reason": "Full periodic synchronization execution was not enabled.",
        }),
        validate=False,
    )
    print("Adversarial smoke mode recorded a plan-only run status; no attack was executed.")


In [ ]:
from commguard.evaluation import evaluate_detector

RUN_ADVERSARIAL_EVALUATION = RUN_FULL_PERIODIC_SYNCHRONIZATION_STUDY
ADVERSARIAL_EVALUATION = None
if RUN_ADVERSARIAL_EVALUATION:
    if ADVERSARIAL_MATRIX is None:
        raise RuntimeError("Collect the approved bounded adversarial matrix first.")
    ADVERSARIAL_EVALUATION = evaluate_detector(
        input_root=ARTIFACTS,
        output=ARTIFACTS,
        benign_extraction_summary=BENIGN_EXTRACTION_SUMMARY,
        adversarial_extraction_summary=ADVERSARIAL_MATRIX["feature_extraction_summary"],
        adversarial_holdout_plan=HOLDOUT_PLAN,
        release_final_adversarial_holdout=False,
    )
    print(ADVERSARIAL_EVALUATION["heldout_adversarial_families"])


In [ ]:
from commguard.artifacts import materialize_adversarial_package

if DEVELOPMENT_SMOKE_ONLY:
    ADVERSARIAL_PACKAGE = ARTIFACTS / "prototype/adversarial"
elif ADVERSARIAL_MATRIX is None or ADVERSARIAL_EVALUATION is None:
    raise RuntimeError("Run and evaluate the approved periodic synchronization study first.")
else:
    ADVERSARIAL_PACKAGE = materialize_adversarial_package(
        ARTIFACTS,
        ADVERSARIAL_MATRIX,
        ADVERSARIAL_EVALUATION,
        environment=ENVIRONMENT,
        provenance={**CONTEXT.to_dict(), **INSTALL_PROVENANCE},
        notebook_filename=NOTEBOOK_FILENAME,
        notebook_sha256=EXPECTED_NOTEBOOK_SHA256,
        configuration={
            "strategy": "periodic_synchronization_local_update_training",
            "synchronization_intervals": list(SYNCHRONIZATION_INTERVALS),
            "detector_frozen_before_adversarial_scoring": True,
        },
    )
print({"machine_readable_package": str(ADVERSARIAL_PACKAGE)})


## Results

not executed. No adversarial, evasion, decoy, or final-holdout result is claimed.


In [ ]:
if ADVERSARIAL_MATRIX is None and not DEVELOPMENT_SMOKE_ONLY:
    raise RuntimeError(
        "NEXT STEP: obtain human approval, set RUN_MODE='full' and "
        "ADVERSARIAL_HUMAN_APPROVAL=True, then rerun from a fresh Kaggle session."
    )
from commguard.artifacts import ArtifactStore, sha256_file

ARCHIVE = Path(f"/kaggle/working/commguard-adversarial-redteam-prototype-{NOTEBOOK_RUN_ID}.tar.gz")
ARCHIVE, SHA_FILE = ArtifactStore(ARTIFACTS).export_with_checksum(ARCHIVE)
ARCHIVE_SHA256 = sha256_file(ARCHIVE)
print(f"NEXT STEP: add {ARCHIVE} to a private Kaggle dataset without renaming it.")
print(f"NEXT STEP: copy SHA-256 {ARCHIVE_SHA256} into EXPECTED_INPUT_SHA256 in the evidence index and report.")
print(
    "NEXT STEP: set that notebook's install-source parameters and notebook SHA-256, "
    "then run from the first cell."
)
